In [1]:
from urllib.request import urlopen

import torch
from torch import nn

torch.set_num_threads(2)
torch.manual_seed(7)

url = "https://s3.amazonaws.com/text-datasets/nietzsche.txt"
with urlopen(url, timeout=60) as response:
    text = " ".join(response.read().decode("utf-8").lower().split())

vocab = sorted(set(text))
encode = {char: i for i, char in enumerate(vocab)}
data = torch.tensor([encode[char] for char in text])
windows = data.unfold(0, 49, 24)
split = int(len(windows) * 0.9)
train, test = windows[:split], windows[split+2::5]
embedding = nn.Embedding(len(vocab), 32)
lstm = nn.LSTM(32, 64, batch_first=True)
head = nn.Linear(64, len(vocab))
optimizer = torch.optim.Adam(nn.ModuleList([embedding, lstm, head]).parameters(), lr=0.003)

def forward(tokens, state=None):
    hidden, state = lstm(embedding(tokens), state)
    return head(hidden), state

for epoch in range(10):
    for batch in train[torch.randperm(len(train))[:4096]].split(128):
        optimizer.zero_grad()
        logits, _ = forward(batch[:, :-1])
        loss = nn.functional.cross_entropy(logits.transpose(1, 2), batch[:, 1:])
        loss.backward()
        optimizer.step()

with torch.no_grad():
    logits, _ = forward(test[:, :-1])
    loss = nn.functional.cross_entropy(logits.transpose(1, 2), test[:, 1:])
    prompt = "the meaning of life"
    logits, state = forward(torch.tensor([[encode[char] for char in prompt]]))
    generated = prompt
    for step in range(160):
        token = torch.multinomial((logits[:, -1] / 0.75).softmax(dim=-1), 1)
        generated += vocab[token.item()]
        logits, state = forward(token, state)

print(f"Holdout character perplexity: {loss.exp():.2f}")
print(generated)

Holdout character perplexity: 7.24
the meaning of lifed the lacte reas whim, momthal tisice onde anding the moll acstion to evil even and in or highint, inditherd, a sister a se as no the morald and exenowess in th
